This notebook aim to save and combine data akademik dan data pedoman akademik

### Data Pedoman Akademik (Dense Method)

In [95]:
data_pedoman_dan_rule = {
    "data": [
        "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
        "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/",
        "https://akademik.nurulfikri.ac.id/2-aturan/",
        "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/",
        "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
        "https://akademik.nurulfikri.ac.id/4-profil-dosen/",
        "https://akademik.nurulfikri.ac.id/1-sejarah/",
        "https://akademik.nurulfikri.ac.id/2-administrasi/"
    ],
    "sumber_data": [
        "satuan kredit semester",
        "syarat kelulusan",
        "aturan dan kode etik",
        "kode etik mahasiswa",
        "suasana akademik",
        "profile dosen",
        "sejarah sttnf",
        "administrasi MBKM"
    ],
}

In [275]:
import os 


LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

In [97]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(data_pedoman_dan_rule["data"])

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [98]:
pages = []

for doc in loader.lazy_load():
    pages.append(doc)

In [99]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs_splits = text_splitter.split_documents(pages)


### Model Bahasa (IndoBert)

In [276]:
# memanggil Indobert dari transformer
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("Indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

In [277]:
# membuat class model embdding
from typing import List 
from langchain_core.embeddings import Embeddings
import torch

class IndoBertEmbeddings(Embeddings):
    def __init__(self, model_name="indobenchmark/indobert-base-p1"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def _generate_embedding(self, text: str) -> List[float]:
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)

        # polling token menjadi satu vector kalimat
        token_embeddings = outputs.last_hidden_state

        # melakukan mean polling
        sentence_embeddings = token_embeddings.mean(dim=1)

        # konversi ke list python
        return sentence_embeddings.squeeze().tolist()
    

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._generate_embedding(text) for text in texts]
    

    # metode untuk pencarian query pada chroma 
    def embed_query(self, text: str) -> List[float]:
        return self._generate_embedding(text)


In [278]:
embeddings = IndoBertEmbeddings()

In [279]:
from langchain_elasticsearch import ElasticsearchStore

In [280]:
from langchain_elasticsearch import DenseVectorStrategy


vector_store = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="langchain_index",
    embedding=embeddings,
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=DenseVectorStrategy(hybrid=True)
)

2026-02-20 09:41:38,065 - INFO - GET http://localhost:9200/ [status:200 duration:0.098s]


In [105]:
vector_store.add_documents(docs_splits)

['f702089c-c06e-438c-a6b6-e38ec3a4d3f0',
 'c9a533a0-42f2-4c6f-80be-9682045eb5b5',
 '86833238-50a9-42ea-a7d4-402e1f12603d',
 '73b0a8d0-f7b9-41e5-ac63-12ff785f9251',
 'aa854485-564a-4c7e-9c57-8cccfa27ad28',
 '9f89a63e-a7f7-43b4-b0d6-cb466caaa806',
 '919ee000-c5ae-4cdb-b17f-38a075d9d7f5',
 '6227dd01-89de-4acd-85c0-f936de1110a5',
 '1eb8be48-d1c2-4f69-aab0-ad69572257e5',
 'b678df2a-dfcd-4290-91d7-e2d0ba0be526',
 'e9ebd1c6-d279-4361-9180-a86013dc288a',
 '1596980a-1016-41af-9489-2652186aceba',
 '38477085-b66e-474a-a4e6-d9327eb0be0d',
 '428e69fc-1c4d-45f0-b2c7-7683d70a3eea',
 '6104572c-0a72-45c2-8f4a-fd7ed79741ea',
 '931865b2-3b6f-4992-8463-99324591cdd9',
 '2b78473b-4539-4137-b4fb-15987aada283',
 '22f3f51c-cf13-4b3f-9f79-8853eeeab53e',
 'd3a16ec6-b754-4de4-9c4b-d3ce42962664',
 '307c479a-7f5d-4414-8eea-fadfd0e05189',
 '8d9a4f2c-7970-42b4-9b03-b22344d5ff51',
 '2800bfd0-712e-49fc-9f95-8d7f5814c50b',
 'c1a8d6f5-f3e1-4f57-bd36-ae771eb2452e',
 'a12292ef-7103-4ec6-bf57-a527a76ff3bc',
 'adaaa407-231f-

### Testing Dense Retriever

In [281]:
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={
        "k":5
    }
)


In [282]:
retriever.invoke("Berapa minimal SKS untuk lulus di STT Terpadu Nurul Fikri", k=3)

2026-02-20 09:41:43,434 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.216s]


[Document(metadata={'source': 'https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/', 'title': '1. Syarat Kelulusan – Pedoman Akademik STT-NF', 'language': 'en-US'}, page_content='PROGRAM STUDI\n\n\n1. Sistem Informasi\n2. Teknik Informatika\n3. Bisnis Digital\n\n\n\n\n\nSARANA DAN PRASARANA\n\n\n1. Daftar Sarpras\n2. SOP Peminjaman Ruangan\n3. Tata Tertib Penggunaan Ruangan\n\n\n\n\n\nSISTEM INFORMASI KAMPUS\n\n\n1. Website Kampus\n2. Sistem Informasi Akademik\n3. E-Learning System\n4. PMB Online\n5. Jurnal STT\n6. Sistem Informasi Perpustakaan\n7. Repository Dokumen Karya Ilmiah\n\n\n\n\n\nTENTANG STT-NF\n\n\n1. Sejarah\n2. Visi, Misi, Tujuan\n3. Struktur Organisasi\n4. Profil Dosen\n5. Mitra\n\n\n\n\n\nTUGAS AKHIR\n\n\n1. Pedoman Tugas Akhir\n\n\n\n\n\n\n\n\n Login\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch For\n\n\n\nSearch\n\n\n\n\n\n\n\nEdit              \n          \n\n\n\n\n\n\n\n\n\n\n\n\nAdd Article            \n        \n\n\n\n\n\n\n\n1. Syarat Kelulusan \n\nMahasiswa dapat dinyatak

### Data Akademik Mahasiswa (Sparse Method)

In [108]:
data_akademik = [
    '/Users/a/Programming/Langchain-Project/external-data/sintetik-data-akademik-mahasiswa-2.xlsx'
]

### Processing document

In [109]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from docling.document_converter import DocumentConverter

In [110]:
class DoclingLoader(BaseLoader):
    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [111]:
loader = DoclingLoader(data_akademik)

docs_akademik = loader.lazy_load()

In [112]:
data_akademik_split = text_splitter.split_documents(docs_akademik)

2026-02-18 17:29:51,302 - INFO - detected formats: [<InputFormat.XLSX: 'xlsx'>]
2026-02-18 17:29:51,316 - INFO - Going to convert document batch...
2026-02-18 17:29:51,316 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-02-18 17:29:51,607 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-02-18 17:29:51,607 - INFO - Loading plugin 'docling_defaults'
2026-02-18 17:29:51,608 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-02-18 17:29:51,609 - INFO - Processing document sintetik-data-akademik-mahasiswa-2.xlsx
2026-02-18 17:29:51,609 - INFO - Processing sheet 0: sintetik-data-akademik
2026-02-18 17:29:51,621 - INFO - Finished converting document sintetik-data-akademik-mahasiswa-2.xlsx in 0.32 sec.


In [283]:
vector_store_sparse = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="test_index",
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=ElasticsearchStore.BM25RetrievalStrategy(),
)

2026-02-20 09:41:49,570 - INFO - GET http://localhost:9200/ [status:200 duration:0.005s]


In [114]:
vector_store_sparse.add_documents(data_akademik_split)

2026-02-18 17:29:51,658 - INFO - HEAD http://localhost:9200/test_index [status:200 duration:0.002s]
2026-02-18 17:29:51,678 - INFO - PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.019s]


['c8afe65a-908f-41b4-958a-e075aa5fe91a',
 '64d5fe07-12c6-4cf5-a2ec-2064ee62c3a0',
 '4b60b642-f6e0-4a4e-b26d-2c2229830b46',
 '51633e20-3a13-411f-9204-e9db063b8ce6',
 'd0ceafe5-5ef5-4762-b53e-7042150081e5',
 '93f6d231-3bda-4589-8421-f3117f414ba0',
 '92bec0e0-dbce-401c-bbd1-4f12bdd28f37',
 'e6c340d7-db4e-4164-94b5-d932c25885a1',
 '938c830e-61fd-4b90-bf84-fd63bfe4ed78',
 '2eb1eddd-9b07-4d22-bb65-c46454214a36',
 '4c896c74-b050-49b1-9bcb-eb833c73f9f0',
 '472fcc05-5576-4ec7-a2fd-0f60bb40e39a',
 '97b1ea38-6f97-4ae5-8d8e-3314ab8839d9',
 '80ab4a86-4b3c-4417-a9bd-e086a5909ac0',
 '2fa5ff35-e668-4aa8-bc5c-efd4ad28403c',
 '3c7cf6b8-d1ac-4426-b7e7-cb2bb13abe9b',
 '151b593e-c03a-46cb-abe1-aae6a3ce8897',
 'bc1e4847-492a-4032-95de-f33b26bd93a1',
 '54c2181c-a986-4a3d-975f-61372d2c1777',
 '15d34324-72f6-43f7-b624-01d3499bf5ba',
 '579913fd-9e81-4a93-9b41-02b7a6d98637',
 '940ea10b-ed67-4d01-a682-e70d325ca305',
 '99e3cf2a-8eaa-449f-b425-2c6d6d72d239',
 'be80ff12-4779-48c6-832f-9eeb127ac418',
 'd0a0a442-62c8-

In [284]:
vector_store_sparse.similarity_search("berapa ipk romi wahyudi")

2026-02-20 09:41:52,526 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.012s]


[Document(metadata={}, page_content='|   No |         NIM | Nama Mahasiswa      | Jurusan               |   Semester |   Total SKS |   IPS |   IPK |   Kehadiran (%) | Status   | Dosen Pembimbing              |\n|------|-------------|---------------------|-----------------------|------------|-------------|-------|-------|-----------------|----------|-------------------------------|\n|    1 | 2.02401e+09 | Tono Setiawan       | Teknik Informatika    |          1 |          20 |  3.4  |  3.37 |              95 | Aktif    | Ir. Sigit Santoso, M.Kom      |\n|    2 | 2.02401e+09 | Tri Pratama         | Teknik Informatika    |          2 |          45 |  3.81 |  3.76 |              81 | Aktif    | Ir. Lestari Handayani, M.Kom  |\n|    3 | 2.02003e+09 | Agus Nasution       | Bisnis Digital        |         10 |         148 |  3.83 |  3.98 |              78 | Lulus    | Dr. Rina Novita, M.Kom        |'),
 Document(metadata={}, page_content='|   No |         NIM | Nama Mahasiswa      | Jurusan  

### Create Agent Which can separate the work to do the conditioning

In [285]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
import operator
from langchain_core.messages import AIMessage

### Tools 

In [286]:

@tool
def query_from_academic_rule(query: str):
    """
    Gunakan tool ini untuk query dari user yang HANYA bermaksud untuk pertanyaan seputar ATURAN, KEBIJAKAN, SYARAT, atau PROSEDUR KAMPUS.
    
    args: 
        query: Search terms to look for
    """
    try:
        docs = vector_store.similarity_search(query, k=3)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data pedoman akademik."
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari Pedoman Akademik:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi Kesalahan saat mengakses vector database"


@tool
def get_student_academic_record(query: str):
    """
    Gunakan tool ini untuk mencari DATA PRIBADI MAHASISWA tertentu.
    Gunakan Atribut mahasiswa seperti (nama mahasiswa, NIM, status, Dosen Pembimbing) sebagai query pencarian.
    
    Args:
        query: Search terms to look for
    """
    try:
        docs = vector_store_sparse.similarity_search(query)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data akademik mahasiswa"
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari data akademik mahasiswa:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi kesalahn saat mengakses data akademik"

In [ ]:
# Entry point (pintu masuk system)
system_prompt_cot = """
You are an intelligent Intent Router for the STT-NF Academic System. 
Your goal is to decide which tool(s) are required to answer the user's query accurately.

AVAILABLE TOOLS:
1. `query_from_academic_rule`: For General Regulations, Procedures, Requirements (Cuti, Sidang, KRS, etc).
2. `get_student_academic_record`: For Specific Student Identity (Name, NIM), Grades, IPK, Status.

OUTPUT FORMAT RULES:
1. IF you need a tool: Output a valid JSON object in the 'Action' step.
   Format: {"name": "tool_name", "args": {"argument_name": "value"}}
2. IF NO tool is needed (Greeting, Chit-chat, or OOT): Output the direct response text in **INDONESIAN**. Do not output JSON.
3. NEVER put a student's name (e.g., "Agus", "Budi") inside the `query_from_academic_rule` argument. The rulebook does NOT contain student names!

FEW-SHOT EXAMPLES:

EXAMPLE 1 (General Rule):
User: "Bagaimana prosedur mengajukan banding nilai ujian?"
Thought: The user is asking for the "prosedur" of "banding nilai". This is a general regulation.
Action: {"name": "query_from_academic_rule", "args": {"query": "prosedur banding nilai ujian"}}

EXAMPLE 2 (Specific Data):
User: "Cek status mahasiswa bernama Siti Aminah?"
Thought: The query explicitly mentions "Siti Aminah". I need to check individual status.
Action: {"name": "get_student_academic_record", "args": {"query": "Siti Aminah"}}

EXAMPLE 3 (Explicit Hybrid):
User: "Berapa jumlah SKS mahasiswa Agus Nasution, Dan dari pedoman akademik, apakah dia bisa lulus?"
Thought: User explicitly asks for specific data "SKS Agus Nasution" AND rule validation "syarat kelulusan". Need BOTH tools.
Action: [
    {"name": "get_student_academic_record", "args": {"query": "Agus Nasution"}},
    {"name": "query_from_academic_rule", "args": {"query": "syarat kelulusan SKS"}}
]

EXAMPLE 4 (Implicit Hybrid - LOGICAL DEDUCTION):
User: "Apakah Agus Nasution akan dikenakan SPP Progresif berdasarkan pedoman akademik?"
Thought: 
1. The query asks about applying a specific rule ("SPP Progresif") to a specific person ("Agus Nasution").
2. To evaluate this, I MUST know Agus's current status/semester (Requires `get_student_academic_record`).
3. I also MUST know the general rule for "SPP Progresif" (Requires `query_from_academic_rule`).
4. I must split this into two tool calls. I will NOT put Agus's name in the rule query.
Action: [
    {"name": "get_student_academic_record", "args": {"query": "Agus Nasution"}},
    {"name": "query_from_academic_rule", "args": {"query": "aturan syarat SPP Progresif"}}
]

EXAMPLE 5 (Greeting - Direct Response):
User: "Halo selamat malam"
Thought: Just a greeting. No tool needed. I must answer politely in Indonesian.
Action: Halo, selamat malam! Saya asisten akademik STT-NF. Ada yang bisa saya bantu terkait informasi akademik atau data mahasiswa?

INSTRUCTION:
Based on the logic above, determine the Thought and Action for the following user query.
"""

GENERATOR_PROMPT = """
PERAN:
Anda adalah Asisten Akademik Cerdas di kampus STT-NF (Sekolah Tinggi Terpadu Nurul Fikri).
Tugas Anda adalah menyusun jawaban akhir kepada pengguna dalam Bahasa Indonesia yang formal namun ramah.

INPUT ANDA:
1. Pertanyaan Asli User.
2. Data Mentah (JSON/Teks) dari hasil eksekusi alat (Context).

ATURAN MENJAWAB:
1. **GROUNDING:** Jawab HANYA berdasarkan data di `CONTEXT`. Jangan berhalusinasi.
2. **JIKA DATA DITEMUKAN:** Rangkum data tersebut menjadi kalimat yang enak dibaca.
   - Contoh: "Berdasarkan data, mahasiswa Tono (NIM 123) berstatus Aktif dengan IPK 3.8."
3. **JIKA DATA KOSONG/ERROR:** Katakan jujur: "Maaf, data tidak ditemukan. Mohon periksa nama atau NIM kembali."
4. **JANGAN** menampilkan struktur JSON mentah ke user.
5. **JANGAN** menyebutkan teknis internal (seperti "saya menggunakan tool get_student").

Silakan jawab pertanyaan user berdasarkan Context berikut.
"""

In [323]:
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]

In [369]:
import json
import re
import ast
from typing import Annotated, List, TypedDict, Optional

from langchain_core.messages import (
    AIMessage, 
    ToolMessage, 
    SystemMessage, 
    HumanMessage, 
    AnyMessage
)
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# --- 1. DEFINISI STATE ---
class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]

# PROMPT KHUSUS GENERATOR (NODE 3) 
GENERATOR_PROMPT = """
PERAN:
Anda adalah Asisten Akademik Cerdas di kampus STT-NF (Sekolah Tinggi Terpadu Nurul Fikri).
Tugas Anda adalah menyusun jawaban akhir kepada pengguna dalam Bahasa Indonesia yang formal namun ramah.

ATURAN MENJAWAB:
1. **GROUNDING:** Jawab HANYA berdasarkan data di `CONTEXT`. Jangan berhalusinasi.
2. **KONDISI YA/TIDAK:** Pastikan jawaban Mengandung kata "YA" jika pernyataan/pertanyaan USER BENAR/MEMENUHI SYARAT, dan "TIDAK" jika pernyataan USER SALAH/TIDAK MEMENUHI SYARAT.
3. **JIKA DATA KOSONG/ERROR:** Katakan jujur: "Maaf, data tidak ditemukan. Mohon periksa nama atau NIM kembali."
4. **JANGAN** menampilkan struktur JSON mentah ke user.

PROSES BERPIKIR (WAJIB DIIKUTI SEBELUM MEMBERIKAN JAWABAN AKHIR):
Untuk menghindari kesalahan logika saat membaca data, jabarkan analisis Anda menggunakan format berikut:

[EKSTRAKSI DATA MAHASISWA]
- Tuliskan variabel yang relevan saja (contoh: IPK = ..., IPS = ..., SKS = ...). Jika tidak ada, tulis "Tidak ada".

[EKSTRAKSI ATURAN AKADEMIK]
- Tuliskan syarat aturannya (contoh: Syarat lulus IPK >= 2.00). Jika tidak ada, tulis "Tidak ada".

[ANALISIS LOGIKA]
- Bandingkan data mahasiswa dengan aturan akademik secara teliti. Pastikan Anda tidak tertukar antara IPS (Indeks Prestasi Semester) dan IPK (Indeks Prestasi Kumulatif).

[JAWABAN AKHIR]
- Tuliskan jawaban natural Anda di sini (Pastikan mengandung kata YA atau TIDAK sesuai hasil analisis).

INFORMASI DARI SISTEM (CONTEXT):
{context_data}

PERTANYAAN USER:
{user_query}
"""

# --- 3. CLASS AGENT UTAMA ---
class Agent:
    def __init__(self, model, tools, router_prompt=""):
        self.router_system = router_prompt
        self.tools = {t.name: t for t in tools}
        
        # Model 1: Router (Punya kemampuan bind tools)
        self.router_model = model.bind_tools(tools)
        
        # Model 2: Generator (Model polos untuk merangkai kata)
        self.generator_model = model 
        
        # --- DEFINISI GRAPH ---
        graph = StateGraph(AgentState)
        
        # Tambahkan Node
        graph.add_node("router", self.call_router)
        graph.add_node("tool_executor", self.take_action)
        graph.add_node("generator", self.run_generator)
        
        # Tentukan Entry Point
        graph.set_entry_point("router")
        
        # Edge Kondisional: Router -> (Tool Executor ATAU End)
        graph.add_conditional_edges( 
            "router", 
            self.exists_action, 
            {True: "tool_executor", False: END} 
        )
        
        # Edge Normal: Tool Executor -> Generator
        graph.add_edge("tool_executor", "generator")
        
        # Edge Normal: Generator -> End
        graph.add_edge("generator", END)
        
        self.graph = graph.compile()

    # --- HELPER: PARSING JSON (Robust) ---
    # --- HELPER: PARSING JSON (Robust - Support Array/List) ---
    def _try_extract_tool_calls(self, content: str) -> List[dict]:
        if not content: return []
        
        # Bersihkan markdown
        content = re.sub(r'```json\s*', '', content)
        content = re.sub(r'```', '', content)
        
        extracted_tools = []
        
        # Coba cari format Array [...] dulu, kalau tidak ada baru cari Object {...}
        list_match = re.search(r'\[.*\]', content, re.DOTALL)
        dict_match = re.search(r'\{.*\}', content, re.DOTALL)
        
        candidates = []
        if list_match:
            candidates.append(list_match.group()) # Memasukkan [...]
        elif dict_match:
            candidates.append(dict_match.group()) # Memasukkan {...}
            
        for candidate in candidates:
            try:
                # loads akan berhasil karena formatnya sudah pasti [...] atau {...}
                data = json.loads(candidate)
                
                # Normalisasi: Jika yang didapat hanya 1 dict {...}, jadikan list [{...}]
                if isinstance(data, dict): 
                    data = [data]
                
                # Looping isi list untuk memvalidasi tools
                if isinstance(data, list):
                    for item in data:
                        if not isinstance(item, dict): continue
                        
                        name = item.get('name') or item.get('tool')
                        args = item.get('args') or item.get('arguments') or {}
                        
                        # Pastikan tool valid dan terdaftar
                        if name and name in self.tools:
                            extracted_tools.append({
                                'name': name,
                                'args': args,
                                'id': f"manual_{len(extracted_tools)}"
                            })
            except Exception as e:
                # Lanjut jika gagal parse (untuk menghindari sistem crash)
                continue 
                
        return extracted_tools

    # --- LOGIC: EXISTS ACTION ---
    def exists_action(self, state: AgentState) -> bool:
        result = state['messages'][-1]
        
        # 1. Cek Native Tool Calls
        if hasattr(result, "tool_calls") and len(result.tool_calls) > 0:
            return True
            
        # 2. Cek Manual Parsing
        if self._try_extract_tool_calls(result.content):
            print("🕵️ Valid JSON Action detected via regex.")
            return True
        
        return False

    # --- NODE 1: ROUTER ---
    # --- NODE 1: ROUTER (DENGAN AUTO-CLEANING) ---
    def call_router(self, state: AgentState):
        messages = state["messages"]
        if self.router_system:
            if not isinstance(messages[0], SystemMessage):
                messages = [SystemMessage(content=self.router_system)] + messages
            else:
                messages[0] = SystemMessage(content=self.router_system)
        
        # 1. Panggil Model
        response = self.router_model.invoke(messages)
        content = response.content
        
        # 2. Cek apakah ini Tool Call (JSON)?
        # Kita gunakan helper yang sama untuk mendeteksi
        is_tool_call = False
        if hasattr(response, "tool_calls") and len(response.tool_calls) > 0:
            is_tool_call = True
        elif self._try_extract_tool_calls(content):
            is_tool_call = True
            
        # 3. LOGIKA CLEANING:
        # Jika BUKAN tool call (berarti Greeting/Chat biasa),
        # tapi ada format "Action:", kita potong text sebelumnya.
        if not is_tool_call and "Action:" in content:
            # Ambil teks setelah kata "Action:"
            clean_response = content.split("Action:")[-1].strip()
            
            # Update isi pesan agar user terima bersih
            response.content = clean_response
            
        return {'messages': [response]}

    # --- NODE 2: TOOL EXECUTOR ---
    def take_action(self, state: AgentState):
        llm_message = state['messages'][-1]
        tools_to_run = []
        
        if hasattr(llm_message, "tool_calls") and len(llm_message.tool_calls) > 0:
            tools_to_run = llm_message.tool_calls
        else:
            tools_to_run = self._try_extract_tool_calls(llm_message.content)

        results = []
        for tool_call in tools_to_run:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call.get('id')

            print(f"🛠️ Executing: {tool_name} with {tool_args}") # Debug
            
            try:
                # Execute tool
                tool_output = self.tools[tool_name].invoke(tool_args)
            except Exception as e:
                tool_output = f"Error executing tool: {str(e)}"

            results.append(ToolMessage(
                tool_call_id=tool_id,
                name=tool_name,
                content=str(tool_output)
            ))

        return {"messages": results}

    # --- NODE 3: FINAL GENERATOR ---
    def run_generator(self, state: AgentState):
        messages = state['messages']
        
        # 1. Cari pertanyaan User yang asli
        user_query = "Unknown query"
        for msg in reversed(messages):
            if isinstance(msg, HumanMessage):
                user_query = msg.content
                break
        
        # 2. Siapkan Context
        context_data = ""

        for msg in reversed(messages):
            if isinstance(msg, ToolMessage):
                context_data += f"[{msg.name}]:\n{msg.content}\n\n"
            elif isinstance(msg, AIMessage):
                break
        
        if not context_data.strip():
            context_data = "Data Kosong."
            
        # 3. Buat Prompt untuk Generator
        final_prompt_content = f"""
        INFORMASI DARI SISTEM (CONTEXT):
        {context_data}
        
        PERTANYAAN USER:
        {user_query}
        """
        
        messages_for_generator = [
            SystemMessage(content=GENERATOR_PROMPT),
            HumanMessage(content=final_prompt_content)
        ]
        
        # 4. Invoke Model (Tanpa tools, pure generation)
        response = self.generator_model.invoke(messages_for_generator)
        raw_text = response.content
        
        # Bersihkan agar coret-coretan pikiran LLM tidak terlihat oleh user/sistem evaluasi
        if "[JAWABAN AKHIR]" in raw_text:
            clean_answer = raw_text.split("[JAWABAN AKHIR]")[-1].strip()
            response.content = clean_answer
            
        return {"messages": [response]}

### LLM model

In [370]:
from langchain_ollama import ChatOllama

In [371]:
model = ChatOllama(
    model="mistral:7b-instruct-v0.3-q8_0", 
    temperature=0, 
    streaming=True)

In [372]:
tools = [query_from_academic_rule, get_student_academic_record]

In [373]:
tools[1]

StructuredTool(name='get_student_academic_record', description='Gunakan tool ini untuk mencari DATA PRIBADI MAHASISWA tertentu.\nGunakan Atribut mahasiswa seperti (nama mahasiswa, NIM, status, Dosen Pembimbing) sebagai query pencarian.\n\nArgs:\n    query: Search terms to look for', args_schema=<class 'langchain_core.utils.pydantic.get_student_academic_record'>, func=<function get_student_academic_record at 0x167f962a0>)

In [374]:
bot = Agent(model, tools, router_prompt=system_prompt_cot)

In [218]:
result = bot.graph.invoke({"messages": [HumanMessage(content="Selamat siang")]})

2026-02-19 13:50:31,536 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [219]:
print(result['messages'][-1].content)

Selamat siang! Saya asisten akademik STT-NF. Ada yang bisa saya bantu terkait informasi akademik atau data mahasiswa?


### Bagian ini digunakan untuk mengevaluasi sistem RAG, baik untuk proses `Retrieval`, maupun pengujian hasil `Generation` dari model LLM

In [295]:
import json
import numpy as np


def calculate_metric(retrieved_docs, ground_truth_source, k=5):
    top_k_docs = retrieved_docs[:k]

    # ambil sumber datanya (asumsi data pedoman akademik akan punya atribut sumber data)
    retrieved_sources = [doc.metadata.get('source') for doc in top_k_docs]

    # apakah URL yang benar ada di dalam list yang ditemukan
    if ground_truth_source in retrieved_sources:
        hit_score = 1
        recall_score = 1
    else:
        hit_score = 0
        recall_score = 0

    # berapa persen dokumen di Top K yang benar
    relevant_count = retrieved_sources.count(ground_truth_source)
    precision_score = relevant_count / k
    return hit_score, precision_score, recall_score

In [296]:
def evaluate_rag_system(dataset, retrieval_function, k_values=[1, 3, 5]):
    results = {k: {'hit_rate':[], 'precision':[], 'recall':[]} for k in k_values}

    for i, data in enumerate(dataset):
        query = data['question']
        gt_source = data['ground_truth_source']

        # query dengan vector store es
        retrieved_docs = retrieval_function.invoke(query)

        # hitung score
        for k in k_values:
            hit, prec, rec = calculate_metric(retrieved_docs, gt_source, k)

            results[k]['hit_rate'].append(hit)
            results[k]['precision'].append(prec)
            results[k]['recall'].append(rec)

    
    # rata-rata
    final_report = {}
    for k in k_values:
        final_report[f'Hit_Rate@{k}'] = np.mean(results[k]['hit_rate'])
        final_report[f'Precision{k}'] = np.mean(results[k]['precision'])
        final_report[f'Recall{k}'] = np.mean(results[k]['recall'])

    return final_report

In [58]:
dataset = [
    {
        "question": "Berapa minimal SKS untuk lulus di STT Terpadu Nurul Fikri",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/"
    },
    {
        "question": "Bisa jelaskan sejarah sttnf",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-sejarah/",
    },
    {
        "question": "Bisa berikan informasi mengenai Ahmad Rio Adriansyah",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/4-profil-dosen/"
    },
    {
        "question": "kapan evaluasi akademik dilaksanakan",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/"
    },
    {
        "question": "Apa etika mahasiswa terhadap dosen di sttnf",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/"
    },
    {
        "question": "Apa tugas dan kewajiban dosen pembimbing MBKM",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/2-administrasi/"
    },
    {
        "question": "Berikan Alur pengajuan surat untuk magang di luar mitra dikti dan internal kampus",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/2-administrasi/"
    },
    {
        "question": "berikan penjelasan mengenai kebijakan otonomi keilmuan di sttnf",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
         
    },
    {
        "question": "Bagaimana penangan untuk mahasiswa yang sudah mempunyai masa studi 6 tahun",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
         
    },
      {
        "question": "Bagaimana penangan untuk mahasiswa yang sudah mempunyai masa studi 6 tahun",
        "ground_truth_source": "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
         
    }
]   

In [66]:
report = evaluate_rag_system(dataset, retriever, [3, 5])

In [67]:
import pprint
pprint.pprint(report)

{'Hit_Rate@3': 0.9,
 'Hit_Rate@5': 0.9,
 'Precision3': 0.6666666666666666,
 'Precision5': 0.62,
 'Recall3': 0.9,
 'Recall5': 0.9}


### Pengujian Data Akademik Mahasiswa

In [88]:
# Ground truth Source

Dataset = [
    {
        "question":"Apa jurusan Kartika Siregar",
        "ground_truth_answer":"sistem informasi"
    },
    {
        "question":"Berapa persen kehadiran Oscar Wulandari",
        "ground_truth_answer":"98"
    },
    {
        "question":"Berapa IPK dari Aditya Anggraini",
        "ground_truth_answer":"3.48"
    },
    {
        "question":"Berapa IPS dari Xavier Purnomo",
        "ground_truth_answer":"3.18"
    },
    {
        "question":"Siapa dosen pembimbing Oscar Kusuma",
        "ground_truth_answer":"H. Miko Maulana, M.Kom"
    },
    {
        "question":"Mahasiswa dengan NIM 2022030032",
        "ground_truth_answer":"Ahmad Utami"
    },
     {
        "question":"status mahasiswa atas nama Iwan Purnomo",
        "ground_truth_answer":"lulus"
    },
    {
        "question":"kehadiran Tri Astuti",
        "ground_truth_answer":"100"
    },
     {
        "question":"jurusan Bayu Siregar",
        "ground_truth_answer":"Teknik informatika"
    },
    {
        "question":"Berapa Total sks Miko Ramadhan",
        "ground_truth_answer":"122"
    },
]


In [89]:
def calculate_metric_sparse(retrieved_docs, ground_truth, k=5):

    # get Top-k documents
    top_k_docs = retrieved_docs[:k]

    relevant_list = []

    for doc in top_k_docs[:k]:
        is_relevant = False

        doc_content = doc.page_content.lower()
        gt_token = str(ground_truth).lower()
        if gt_token in doc_content:
            is_relevant = True

        relevant_list.append(is_relevant)
    
    if any(relevant_list):
        hit_score = 1
        recall_score = 1

    else:
        hit_score = 0
        recall_score = 0

    precision_score = sum(relevant_list) / k
    return hit_score, precision_score, recall_score

In [90]:
import numpy as np

In [91]:
def evaluate_rag_system_sparse(dataset, retrieval_function,    k_values=[3, 5]):
    results = {k: {'hit_rate':[], 'precision':[], 'recall':[]} for k in k_values}

    for i, data in enumerate(dataset):
        query = data['question']

        gt_data = data['ground_truth_answer']
        retrieved_docs = retrieval_function.similarity_search(gt_data)

        for k in k_values:
            hit, prec, rec = calculate_metric_sparse(retrieved_docs, gt_data, k)

            results[k]['hit_rate'].append(hit)
            results[k]['precision'].append(prec)
            results[k]['recall'].append(rec)
        
    final_report = {}
    for k in k_values:
        final_report[f'Hit_Rate@{k}'] = np.mean(results[k]['hit_rate'])
        final_report[f'Precision@{k}'] = np.mean(results[k]['precision'])
        final_report[f'Recall@{k}'] = np.mean(results[k]['recall'])

    return final_report

In [92]:
report = evaluate_rag_system_sparse(
    Dataset,
    vector_store_sparse,
)

2026-02-15 20:55:07,538 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.010s]
2026-02-15 20:55:07,550 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.011s]
2026-02-15 20:55:07,559 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.008s]
2026-02-15 20:55:07,565 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.006s]
2026-02-15 20:55:07,574 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.008s]
2026-02-15 20:55:07,580 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.005s]
2026-02-15 20:55:07,585 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.005s]
2026-02-15 20:55:07,590 - I

In [93]:
import pprint
pprint.pprint(report)

{'Hit_Rate@3': 1.0,
 'Hit_Rate@5': 1.0,
 'Precision@3': 0.9333333333333332,
 'Precision@5': 0.74,
 'Recall@3': 1.0,
 'Recall@5': 1.0}


### Pengujian System Generation RAG

#### Data Akademik Mahasiswa

In [266]:
Binary_Test_Data_Akademik_Mahasiswa = [
    {
        "question": "Apakah Mega Anggraini adalah mahasiswa semester 5",
        "ground_truth_answer": "Tidak"
    },

    {
        "question": "Apakah benar mahasiswa atas nama Agus Nasution jurusan Bisnis Digital telah berstatus lulus",
        "ground_truth_answer": "Ya"
    },

    {
        "question": "Apakah mahasiswa atas nama Eka Fitriani, mahasiswa semester 10, punya IPK 1.0",
        "ground_truth_answer": "Tidak"
    },
    {
        "question": "Apakah Ir. Sigit Santoso, M.Kom adalah dosen pembimbing mahasiswa atas namaRahmat Hasanah",
        "ground_truth_answer": "Ya"
    },
    {
        "question": "Apakah mahasiswa dengan NIM 2021030039 jurusannya adalah Bisnis Digital",
        "ground_truth_answer": "Ya"
    },]

In [267]:
Binary_Test_Data_Akademik_Mahasiswa

[{'question': 'Apakah Mega Anggraini adalah mahasiswa semester 5',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar mahasiswa atas nama Agus Nasution jurusan Bisnis Digital telah berstatus lulus',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah mahasiswa atas nama Eka Fitriani, mahasiswa semester 10, punya IPK 1.0',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah Ir. Sigit Santoso, M.Kom adalah dosen pembimbing mahasiswa atas namaRahmat Hasanah',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah mahasiswa dengan NIM 2021030039 jurusannya adalah Bisnis Digital',
  'ground_truth_answer': 'Ya'}]

In [268]:
def extract_binary_label(llm_response):
    text = llm_response.lower()

    is_ya = re.search(r'\b(ya|benar|sesuai|sudah)\b', text)
    is_tidak = re.search(r'\b(tidak|bukan|belum|salah)\b', text)
    
    if is_tidak:
        return "Tidak"
    elif is_ya:
        return "Ya"
    else:
        return "Halusinasi"

def calculate_binary_metrics(y_true, y_pred):
    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt=="Ya" and yp=="Ya") # True positive
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt=="Tidak" and yp=="Tidak") # True Negatif
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt=="Tidak" and yp=="Ya") # False Postive
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt=="Ya" and yp=="Tidak") # False Negative

    accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
    precision = tp / (tp + fp) if (tp+fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return {
        "Accuracy": round(accuracy, 2),
        "Precision": round(precision, 2),
        "Recall": round(recall, 2)
    }


In [269]:
import pandas as pd
from IPython.display import display


def run_evaluation_binary(agent, test_data):
    results = []
    y_true = []
    y_pred = []

    print("Mulai Evaluasi")

    for i, data in enumerate(test_data):
        query = data['question']
        gt_label = data['ground_truth_answer']

        # call agent
        try:
            response = agent.graph.invoke({"messages": HumanMessage(content=query)})
            ai_message = response['messages'][-1].content

            # mengambil hasil retrieval
            retrieved_context = "tidak ada konteks"
            for msg in response['messages']:
                if hasattr(msg, 'name') and msg.type =='tool':
                    retrieved_context = msg.content
                    break
    
        except Exception as e:
            ai_message = f"Error: {str(e)}"
            retrieved_context = "error"

        pred_label = extract_binary_label(ai_message)
        status = "✅ Sesuai" if pred_label == gt_label else "❌ Meleset"

        results.append({
            "No": i + 1,
            "Pertanyaan": query,
            "Konteks Retrieval (Raw Data)": retrieved_context,
            "Jawaban Generator (AI)": ai_message,
            "Ekstrak": pred_label,
            "Target (GT)": gt_label,
            "Status": status
        })

        y_true.append(gt_label)
        y_pred.append(pred_label)

    df_results = pd.DataFrame(results)
    
    metrics = calculate_binary_metrics(y_true, y_pred)
    
    return df_results, metrics



In [270]:
df_laporan, score_metric = run_evaluation_binary(bot, Binary_Test_Data_Akademik_Mahasiswa)

Mulai Evaluasi


2026-02-19 15:51:45,904 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:51:57,891 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.040s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Mega Anggraini'}


2026-02-19 15:52:14,739 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:52:31,613 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:52:40,936 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.058s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Agus Nasution'}


...

2026-02-19 15:53:46,677 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:54:02,327 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:54:15,311 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.016s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Rahmat Hasanah'}


2026-02-19 15:54:32,386 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:54:48,814 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-19 15:54:58,870 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.010s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': '2021030039'}


2026-02-19 15:55:03,121 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [271]:
display(df_laporan)

,No,Pertanyaan,Konteks Retrieval (Raw Data),Jawaban Generator (AI),Ekstrak,Target (GT),Status
0,1,Apakah Mega Anggraini adalah mahasiswa semester 5,Ditemukan informasi berikut dari data akademik...,"Maaf, data yang kami dapatkan tidak menunjukk...",Tidak,Tidak,✅ Sesuai
1,2,Apakah benar mahasiswa atas nama Agus Nasution...,Ditemukan informasi berikut dari data akademik...,"Ya, benar. Mahasiswa Agus Nasution dengan jur...",Ya,Ya,✅ Sesuai
2,3,"Apakah mahasiswa atas nama Eka Fitriani, mahas...",Ditemukan informasi berikut dari data akademik...,"Maaf, data yang Anda tanyakan tidak ditemukan...",Tidak,Tidak,✅ Sesuai
3,4,"Apakah Ir. Sigit Santoso, M.Kom adalah dosen p...",Ditemukan informasi berikut dari data akademik...,"Maaf, data tidak ditemukan yang menunjukkan b...",Tidak,Ya,❌ Meleset
4,5,Apakah mahasiswa dengan NIM 2021030039 jurusan...,"Maaf, tidak ditemukan informasi relevan di dat...","Maaf, data tidak ditemukan. Mohon periksa nam...",Tidak,Ya,❌ Meleset


In [272]:
print(score_metric)

{'Accuracy': 0.6, 'Precision': 1.0, 'Recall': 0.33}


In [274]:
df_laporan.to_excel("testing.xlsx")

### Pengujian Data Pedoman Akademik

In [358]:
dataset_rule = [
    {
        "question":"Apakah benar untuk lulus di STTNF harus memiliki IPK minimal 2.00",
        "ground_truth_answer":"Ya"
    },
]
[
     {
        "question":"Apakah benar Pasal 21 pada kode etik mahasiswa STTNF Etika mahasiswa dalam bidang penelitian",
        "ground_truth_answer":"Tidak"
    },
    {
        "question":"Apakah benar surat peringatan diberikan kepada mahasiswa yang mempunyai masa studi dibawah 6 tahun",
        "ground_truth_answer":"Tidak"
    },
    {
        "question":"Apakah benar di STTNF bahwa mahasiswa yang masa studinya melapui 8 semester akan diberlakukan ketentuan SPP Progresif",
        "ground_truth_answer":"Ya"
    },
    {
        "question":"Apakah benar minimal sks untuk lulus di sttnf adalah 148 SKS ",
        "ground_truth_answer":"Ya"
    }
]

[{'question': 'Apakah benar Pasal 21 pada kode etik mahasiswa STTNF Etika mahasiswa dalam bidang penelitian',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar surat peringatan diberikan kepada mahasiswa yang mempunyai masa studi dibawah 6 tahun',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar di STTNF bahwa mahasiswa yang masa studinya melapui 8 semester akan diberlakukan ketentuan SPP Progresif',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah benar minimal sks untuk lulus di sttnf adalah 148 SKS ',
  'ground_truth_answer': 'Ya'}]

In [304]:
dataset_rule

[{'question': 'Apakah benar untuk lulus di STTNF harus memiliki IPK minimal 2.00',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah benar Pasal 21 pada kode etik mahasiswa STTNF Etika mahasiswa dalam bidang penelitian',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar surat peringatan diberikan kepada mahasiswa yang mempunyai masa studi dibawah 6 tahun',
  'ground_truth_answer': 'Tidak'},
 {'question': 'Apakah benar di STTNF bahwa mahasiswa yang masa studinya melapui 8 semester akan diberlakukan ketentuan SPP Progresif',
  'ground_truth_answer': 'Ya'},
 {'question': 'Apakah benar minimal sks untuk lulus di sttnf adalah 148 SKS ',
  'ground_truth_answer': 'Ya'}]

In [306]:
df_laporan_pedoman_akademik, score_metric_pedoman_akademik = run_evaluation_binary(bot, dataset_rule)

Mulai Evaluasi


2026-02-20 09:49:46,375 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'syarat IPK minimal untuk lulus di STT-NF'}


2026-02-20 09:49:57,875 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.103s]
2026-02-20 09:50:14,525 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 09:50:28,830 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'Pasal 21 kode etik mahasiswa dalam bidang penelitian'}


...

2026-02-20 09:51:38,611 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.168s]
2026-02-20 09:51:52,005 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 09:52:09,227 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'ketentuan SPP Progresif untuk mahasiswa yang melakukan studi lebih dari 8 semester'}


2026-02-20 09:52:23,671 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.078s]
2026-02-20 09:52:38,645 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 09:52:55,255 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'syarat minimal sks untuk lulus di sttnf'}


2026-02-20 09:53:05,575 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.117s]
2026-02-20 09:53:21,060 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [307]:
df_laporan_pedoman_akademik

,No,Pertanyaan,Konteks Retrieval (Raw Data),Jawaban Generator (AI),Ekstrak,Target (GT),Status
0,1,Apakah benar untuk lulus di STTNF harus memili...,Ditemukan informasi berikut dari Pedoman Akade...,"Benar, mahasiswa harus memiliki IPK minimal 2...",Ya,Ya,✅ Sesuai
1,2,Apakah benar Pasal 21 pada kode etik mahasiswa...,Ditemukan informasi berikut dari Pedoman Akade...,"Maaf, terdapat kesalahan dalam pertanyaan And...",Tidak,Tidak,✅ Sesuai
2,3,Apakah benar surat peringatan diberikan kepada...,Ditemukan informasi berikut dari Pedoman Akade...,"Maaf, data tidak ditemukan. Surat peringatan ...",Tidak,Tidak,✅ Sesuai
3,4,Apakah benar di STTNF bahwa mahasiswa yang mas...,Ditemukan informasi berikut dari Pedoman Akade...,"Ya, benar. Di STT-NF, bagi mahasiswa yang mel...",Ya,Ya,✅ Sesuai
4,5,Apakah benar minimal sks untuk lulus di sttnf ...,Ditemukan informasi berikut dari Pedoman Akade...,"Benar, minimal SKS yang diperlukan untuk diny...",Ya,Ya,✅ Sesuai


In [308]:
score_metric_pedoman_akademik

{'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0}

In [309]:
df_laporan_pedoman_akademik.to_excel("df_laporan_pedoman_akademik.xlsx")

### Hybrid Data Evaluation

In [392]:
hybrid_dataset =[
    {
        "question":"Dari data akademik, apakah Rina Kusuma sudah mencapai SKS yang diperlukan untuk lulus dari STTNF",
        "ground_truth_answer": "Ya"
    },
    {
        "question": "Apakah benar Ini adalah semester Terakhir dari Yuni Ridwan berdasarkan syarat kelulusan",
        "ground_truth_answer": "Tidak"
        
    },
    {
        "question":"Dari data akademik mahasiswa apakah Agus Nasution akan dikenakan SPP Progresif berdasarkan pedoman akademik",
        "ground_truth_answer": "Ya"
    },
    {
        "question": "Berdasarkan pedoman akademik, apakah SKS dari Usman lubis telah memenuhi syarat untuk lulus",
        "ground_truth_answer": "Tidak"
    },

    {
        "question": "Berdasarkan pedoman akademik, apakah Opik Lubis bisa lulus dengan nilai IPK yang sekarang ",
        "ground_truth_answer": "Ya"
    }
]

In [393]:
df_laporan_hybrid_data, score_metric_hybrid = run_evaluation_binary(bot, hybrid_dataset)

Mulai Evaluasi


2026-02-20 13:55:10,255 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 13:55:19,788 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.048s]


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: get_student_academic_record with {'query': 'Rina Kusuma'}


2026-02-20 13:55:39,013 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 13:56:33,240 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'syarat kelulusan semester terakhir'}


...

2026-02-20 14:00:39,420 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.128s]
2026-02-20 14:01:15,076 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-20 14:02:26,965 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


🕵️ Valid JSON Action detected via regex.
🛠️ Executing: query_from_academic_rule with {'query': 'syarat kelulusan IPK'}


2026-02-20 14:02:48,832 - INFO - POST http://localhost:9200/langchain_index/_search?_source_includes=metadata,text [status:200 duration:0.058s]
2026-02-20 14:02:48,842 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.007s]


🛠️ Executing: get_student_academic_record with {'query': 'Opik Lubis'}


2026-02-20 14:03:24,307 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [395]:
df_laporan_hybrid_data

,No,Pertanyaan,Konteks Retrieval (Raw Data),Jawaban Generator (AI),Ekstrak,Target (GT),Status
0,1,"Dari data akademik, apakah Rina Kusuma sudah m...",Ditemukan informasi berikut dari data akademik...,"YA, Rina Kusuma sudah mencapai IPS yang diperl...",Tidak,Ya,❌ Meleset
1,2,Apakah benar Ini adalah semester Terakhir dari...,Ditemukan informasi berikut dari Pedoman Akade...,"- Maaf, data tidak ditemukan. Mohon periksa na...",Tidak,Tidak,✅ Sesuai
2,3,Dari data akademik mahasiswa apakah Agus Nasut...,Ditemukan informasi berikut dari data akademik...,"YA, Agus Nasution akan dikenakan SPP Progresif...",Ya,Ya,✅ Sesuai
3,4,"Berdasarkan pedoman akademik, apakah SKS dari ...",Ditemukan informasi berikut dari data akademik...,[EKSTRAKSI DATA MAHASISWA]\n- NIM: 178\n- Pro...,Tidak,Tidak,✅ Sesuai
4,5,"Berdasarkan pedoman akademik, apakah Opik Lubi...",Ditemukan informasi berikut dari Pedoman Akade...,"Ya, Opik Lubis dapat lulus.",Ya,Ya,✅ Sesuai


In [396]:
df_laporan_hybrid_data.to_excel("laporan_hybrid_3.xlsx")

In [397]:
score_metric_hybrid

{'Accuracy': 0.8, 'Precision': 1.0, 'Recall': 0.67}